# 🏨 AGODA Price Crawler

Chạy lần lượt các cell từ trên xuống: **① Cấu hình → ② Đọc input → ③ Crawl → ④ Xem kết quả**.

**Đổi nguồn input:** sửa `INPUT_MODE` ở cell ① — `"gsheet"` (Google Sheet online) hoặc `"offline"` (file CSV/XLSX trên máy).

**Format file offline:** chỉ cần **3 cột đầu theo đúng thứ tự** `hotel_name, hotel_url, room_type` (tên cột không quan trọng, chỉ cần đúng thứ tự). Có file mẫu ở `input/TEMPLATE_hotels.csv`.

**Output** nằm trong `results/agoda/`:
- `FINAL_<YYYYMMDD>.csv` — kết quả cuối
- `TEMP_agoda.csv` — checkpoint: lỡ tắt giữa chừng, chạy lại cell ③ sẽ tự resume phần chưa xong

In [1]:
# ════════════════ ① CẤU HÌNH ════════════════

# ── Nguồn input: "gsheet" (online) hoặc "offline" (file trên máy) ──
INPUT_MODE = "gsheet"

# Dùng khi INPUT_MODE = "gsheet" (gid của tab được tự lấy từ URL)
GSHEET_URL = "https://docs.google.com/spreadsheets/d/1SnpIayEwkMMaLow2ImZLS9enmn7A80LB0IeQMAkemio/edit?gid=1827224235#gid=1827224235"

# Dùng khi INPUT_MODE = "offline" — đường dẫn tuyệt đối, hoặc tương đối so với 31.crawl-tool
# ⚠️ File phải có 3 cột đầu là (tên KS, URL, loại phòng) — file "TEMP_*" là checkpoint OUTPUT, không phải input!
OFFLINE_FILE = "input/v"

# ── Tham số crawl ──
WEEKS      = 6      # số tuần cần crawl
MAX_HOTELS = 0      # 0 = crawl tất cả; đặt 5 để test nhanh 5 khách sạn đầu
SHARD      = ""     # "" = không chia; "1/3" = chạy phần 1 trong 3 phần (chạy lần lượt 1/3, 2/3, 3/3)

In [2]:
# ════════════════ ② ĐỌC INPUT ════════════════
import os, sys

if "ROOT" not in globals():                    # giữ nguyên ROOT khi chạy lại cell
    ROOT = os.path.abspath("")                 # .../31.crawl-tool (nơi đặt notebook này)
assert os.path.isdir(os.path.join(ROOT, "crawler")), (
    f"Không tìm thấy package `crawler` trong {ROOT} — hãy mở notebook từ thư mục 31.crawl-tool")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import crawler
from crawler.hotels_io import read_hotels

if INPUT_MODE == "gsheet":
    INPUT = GSHEET_URL
    print("📡 Input: Google Sheet online")
else:
    INPUT = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(ROOT, OFFLINE_FILE)
    assert os.path.exists(INPUT), f"Không tìm thấy file: {INPUT}"
    print(f"📁 Input: file offline — {INPUT}")

hotels = read_hotels(INPUT)
print(f"✅ Đọc được {len(hotels)} khách sạn. 5 dòng đầu:")
for name, url, room in hotels[:5]:
    print(f"   • {name} — {room}")

📡 Input: Google Sheet online
✅ Đọc được 74 khách sạn. 5 dòng đầu:
   • muong thanh — Phòng Hai Giường Đơn Loại Sang (Deluxe Twin Room)
   • garden plaza — Phòng Superior 2 giường (Superior Twin Room)
   • holiday inn — Phòng Tiêu chuẩn 1 giường King Phù hợp cho người khuyết tật (1 King Standard Accessible)
   • eastin — Phòng Superior (Superior)
   • kin wander tan binh — Phòng Tiêu chuẩn có ban công (Standard with Balcony)


In [3]:
# (TÙY CHỌN) Tải Google Sheet về file offline — lần sau chỉ cần đổi INPUT_MODE = "offline"
import pandas as pd
from crawler.hotels_io import _gsheet_url

os.makedirs(os.path.join(ROOT, "input"), exist_ok=True)
dest = os.path.join(ROOT, "input", "agoda_hotels.csv")
pd.read_csv(_gsheet_url(GSHEET_URL)).to_csv(dest, index=False, encoding="utf-8-sig")
print(f"💾 Đã lưu bản offline: {dest}")

💾 Đã lưu bản offline: /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/input/agoda_hotels.csv


In [4]:
# ════════════════ ③ CRAWL ════════════════
OUTDIR = os.path.join(ROOT, "results", "agoda")
os.makedirs(OUTDIR, exist_ok=True)
os.chdir(OUTDIR)                     # output (FINAL_*.csv, TEMP_agoda.csv) nằm ở đây

kwargs = dict(
    site="agoda",                    # direct replay (nhanh) + Camoufox warm
    input=INPUT,
    weeks=WEEKS,
)
if MAX_HOTELS:
    kwargs["max"] = MAX_HOTELS
if SHARD:
    kwargs["shard"] = SHARD

await crawler.arun(**kwargs)         # notebook cho phép await trực tiếp

📂 Resume: 241 rows from TEMP_agoda.csv
🚀 AGODA crawl | 74 hotels × 6w | direct+fallback | engine=camoufox | W1=2026-07-18
✔️  1/74 muong thanh — complete, skip
✔️  2/74 garden plaza — complete, skip
✔️  3/74 holiday inn — complete, skip
✔️  4/74 eastin — complete, skip
✔️  5/74 kin wander tan binh — complete, skip
✔️  6/74 vissai — complete, skip
✔️  7/74 MV Nguyễn Kiệm — complete, skip
✔️  8/74 MV Cửu Long — complete, skip
✔️  9/74 Dream palace ben thanh — complete, skip
✔️  10/74 jw marriott hotel & suites saigon — complete, skip
✔️  11/74 liberty — complete, skip
✔️  12/74 the myst — complete, skip
✔️  13/74 mia q2 — complete, skip
✔️  14/74 paragon — complete, skip
✔️  15/74 Bến Thành Boutique Hotel — complete, skip
✔️  16/74 Alagon City Hotel & Spa — complete, skip
✔️  17/74 G8 Riverside — complete, skip
✔️  18/74 The Hammock Hotel Fine Arts Museum — complete, skip
✔️  19/74 The Concept Hotel HCMC - D1 — complete, skip
✔️  20/74 A25 — complete, skip
✔️  21/74 Val Solei Hotel — com

'FINAL_20260713.csv'

In [5]:
# ════════════════ ④ XEM KẾT QUẢ ════════════════
import glob
import pandas as pd

OUTDIR = os.path.join(ROOT, "results", "agoda")
files = sorted(glob.glob(os.path.join(OUTDIR, "FINAL_*.csv")))
assert files, "Chưa có file FINAL nào — hãy chạy cell ③ trước."
latest = files[-1]
df = pd.read_csv(latest)
print(f"📄 {latest} — {len(df)} dòng")
df.head(20)

📄 /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/agoda/FINAL_20260713.csv — 241 dòng


,hotel_name,room_type,price_w1,price_w2,price_w3,price_w4,price_w5,price_w6
0,Harvest Day Hoi An - Hotel,Standard Double with Garden View,"2,337,643","1,968,961","2,256,571","2,469,029","1,968,961","2,450,032"
1,Reu Boutique Hotel - Hotel,Superior 2 giường hoặc giường đôi (Superior Tw...,"1,826,898","2,141,683","2,127,112","2,141,777","2,149,601","2,141,196"
2,Renaissance Danang Hoi An Resort & Spa,"Deluxe, Guest room, 2 Twin, Garden view, Balcony","2,500,000","2,400,000","2,185,448","2,400,000","2,400,000","2,300,000"
3,Victoria Hoi An Beach Resort & Spa,Phòng Đôi Hướng Sông (River View Double Room),"2,953,217","3,032,848","2,753,124","3,007,098","7,125,284","2,534,135"
4,KOI Resort and Spa Hoi An,Bungalow Nhìn ra vườn (Bungalow with Garden View),"1,769,110","7,527,968","1,715,762","1,717,345","1,716,605","1,717,001"
5,Hoi An Beach Resort,Phòng Superior (Superior Room),"1,747,831","1,686,054","1,373,347","1,747,160","1,374,792","1,440,649"
6,Palm Garden Beach Resort & Spa,Superior giường đôi hoặc 2 giường đơn Hướng vư...,"2,461,256","3,305,244","3,308,121","2,461,256","2,461,256","2,461,256"
7,ENSO Retreat Hoi An - Rediscovery & Serenity,Phòng Deluxe giường đôi Hướng vườn (Deluxe Dou...,"1,273,074","1,318,801","1,261,610","1,307,337","1,307,466","1,295,873"
8,Sứ Retreat Hoi An - Riverfront Resort,Suite Lớn có Ban Công (Executive Suite with Ba...,"2,703,704","2,424,043","3,218,695","3,306,878","3,306,878","3,306,878"
9,Banla Boutique Hotel,Classic Double Room,"916,513","1,125,220","1,125,220","1,125,220","1,125,220","1,125,220"
